# 尺寸聚類

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/09-anchor-clustering/)

先執行環境格，再閱讀、執行下方完整實驗。各節互相獨立，不需要上一節的 runtime。

本版使用 CPU 驗證機制及短步參數更新；長訓練的辨識效果、AP 與 GPU 效率仍待補測。這些範例沒有預訓練權重，也不需下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.1.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 本輪只驗證 CPU。已裝相同 PyTorch 版本時保留其 CPU/CUDA build。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
import torch


def size_iou(sizes, anchors):
    intersection = torch.minimum(sizes[:,None,:],anchors[None,:,:]).prod(-1)
    return intersection/(sizes.prod(-1)[:,None]+anchors.prod(-1)[None,:]-intersection)


def cluster(sizes, initial, steps=10):
    anchors = initial.clone()
    for _ in range(steps):
        assignment = (1-size_iou(sizes,anchors)).argmin(1)
        updated = anchors.clone()
        for k in range(len(anchors)):
            selected = sizes[assignment==k]
            if len(selected):
                updated[k] = selected.mean(0)
        if torch.allclose(updated,anchors):
            return updated,assignment
        anchors = updated
    return anchors,assignment


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    sizes = torch.tensor([[8.,8.],[9.,8.],[8.,9.],[32.,16.],[30.,16.],[32.,18.]])
    # These six shapes stand for TRAIN annotations after the same input resize.
    anchors, groups = cluster(sizes,sizes[[0,-1]])
    expected = torch.tensor([[25/3,25/3],[94/3,50/3]])
    assert torch.allclose(anchors,expected)
    assert groups.tolist() == [0,0,0,1,1,1]
    baseline = torch.tensor([[16.,16.],[16.,16.]])
    old = size_iou(sizes,baseline).max(1).values.mean()
    new = size_iou(sizes,anchors).max(1).values.mean()
    assert new > old
    median = torch.stack([sizes[groups==k].median(dim=0).values for k in range(2)])
    assert torch.equal(median,torch.tensor([[8.,8.],[32.,16.]]))
    median_fit = size_iou(sizes,median).max(1).values.mean()
    assert abs(median_fit.item()-.9340278) < 1e-6
    assert median_fit > new  # This fixture only; not a general optimal-update theorem.
    # Held-out aspect ratios: neither cluster necessarily fits the new source.
    shifted = torch.tensor([[4.,40.],[40.,4.]])
    shifted_fit = size_iou(shifted,anchors).max(1).values.mean()
    print('groups',groups.tolist(),'anchors pixel',anchors.tolist())
    print('train mean best size IoU',round(old.item(),4),'->',round(new.item(),4))
    print('median anchors',median.tolist(),'train coverage',round(median_fit.item(),4))
    print('new-source shape coverage',round(shifted_fit.item(),4),'not detector AP')
    # Co-scaling the data and anchors leaves size IoU unchanged.
    assert torch.allclose(size_iou(sizes,anchors),size_iou(sizes*2,anchors*2))


if __name__ == '__main__':
    main()


groups [0, 0, 0, 1, 1, 1] anchors pixel [[8.333333015441895, 8.333333015441895], [31.33333396911621, 16.66666603088379]]
train mean best size IoU 0.3817 -> 0.9119
median anchors [[8.0, 8.0], [32.0, 16.0]] train coverage 0.934
new-source shape coverage 0.1975 not detector AP
